In [ ]:
%pip install numpy pandas numba scipy pyarrow huggingface-hub requests plotly

In [ ]:
import os
import sys

REPO_URL  = "https://github.com/payamdav/pycrypto2.git"
REPO_NAME = "pycrypto2"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

REPO_PATH = os.path.abspath(REPO_NAME)
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)

In [ ]:
# Base asset — change here to switch asset for all cells below
asset = "btcusdt"

In [ ]:
import time
from packages.tools.candle_preloader import preload_candles
from packages.tools.candle_cache import preload_asset_candles

t0 = time.perf_counter()

# File cache (download if missing, then read from local parquet)
preload_candles([asset])

# In-memory cache (fast repeated slicing)
preload_asset_candles(asset)

t_preload = time.perf_counter() - t0
print(f"\nCandle pre-load total: {t_preload:.3f}s")

In [ ]:
from packages.tools.metrics_cache import (
    create_metrics_cache_base_file,
    metrics_cache_volume_median_iqr,
    metrics_cache_volume_mean_stddev,
)

t0 = time.perf_counter()

_t = time.perf_counter()
create_metrics_cache_base_file(asset)
print(f"  create_metrics_cache_base_file   {time.perf_counter() - _t:.3f}s")

_t = time.perf_counter()
metrics_cache_volume_median_iqr(asset)
print(f"  metrics_cache_volume_median_iqr  {time.perf_counter() - _t:.3f}s")

_t = time.perf_counter()
metrics_cache_volume_mean_stddev(asset)
print(f"  metrics_cache_volume_mean_stddev {time.perf_counter() - _t:.3f}s")

t_metrics = time.perf_counter() - t0
print(f"\nMetrics cache total: {t_metrics:.3f}s")

In [ ]:
print(f"Total pre-load + metrics time: {t_preload + t_metrics:.3f}s")

In [ ]:
# ── Input parameters (edit here) ──────────────────────────────────────────
look_back        = 1440          # candles in the look-back window (includes anchor)
look_ahead       = 240           # candles in the look-ahead window
k                = 100.0         # price scale factor (ratio 0.01 → 1.0)
bins_count       = 200           # KDE histogram bins over [-1, 1]
bandwidth        = 5             # kernel half-width in bins
kernel_type      = "Triangular"  # "Triangular" | "Epanechnikov" | "Uniform"
kde_ignore_borders = True        # exclude clipped prices (±1.0) from histogram
# ──────────────────────────────────────────────────────────────────────────

In [ ]:
import numpy as np
from datetime import datetime, timezone, timedelta
from packages.tools.candle_cache import get_cached_candles
from strategies.lbla_n_vp.lbla_n_vp import lookback_lookahead_normalized_vp

# Determine usable anchor index range
cache = get_cached_candles(asset)
ts_arr = cache["ts"]
n = cache["_len"]

# Valid anchors: [look_back-1 : n-look_ahead] (inclusive)
valid_start = look_back - 1
valid_end   = n - look_ahead - 1  # inclusive

rng = np.random.default_rng(seed=42)
anchor_idx = int(rng.integers(valid_start, valid_end + 1))
anchor_ts  = int(ts_arr[anchor_idx])
# current_time = anchor_ts + 60_000  (moment after anchor candle closes)
current_ts  = anchor_ts + 60_000
dt_str = datetime.fromtimestamp(current_ts / 1000, tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S")

print(f"Single-call anchor: index={anchor_idx}, datetime={dt_str}")

# Warm-up call to compile numba jitted functions before timing
_ = lookback_lookahead_normalized_vp(
    asset=asset, look_back=look_back, look_ahead=look_ahead,
    datetime=dt_str, k=k, bins_count=bins_count,
    bandwidth=bandwidth, kernel_type=kernel_type,
    kde_ignore_borders=kde_ignore_borders,
)

# Timed single call
data = lookback_lookahead_normalized_vp(
    asset=asset, look_back=look_back, look_ahead=look_ahead,
    datetime=dt_str, k=k, bins_count=bins_count,
    bandwidth=bandwidth, kernel_type=kernel_type,
    kde_ignore_borders=kde_ignore_borders,
)

print("\n── Single-call timing ──")
for fn_name, t in data["timing"].items():
    print(f"  {fn_name:<25} {t*1000:.3f} ms")

In [ ]:
# 100 random-minute calls — averages across steady-state JIT
n_calls = 100
anchor_indices = rng.integers(valid_start, valid_end + 1, size=n_calls)

timing_accum: dict[str, float] = {}

for idx in anchor_indices:
    anchor_ts_i = int(ts_arr[int(idx)]) + 60_000
    dt_i = datetime.fromtimestamp(anchor_ts_i / 1000, tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    result = lookback_lookahead_normalized_vp(
        asset=asset, look_back=look_back, look_ahead=look_ahead,
        datetime=dt_i, k=k, bins_count=bins_count,
        bandwidth=bandwidth, kernel_type=kernel_type,
        kde_ignore_borders=kde_ignore_borders,
    )
    for fn_name, t in result["timing"].items():
        timing_accum[fn_name] = timing_accum.get(fn_name, 0.0) + t

print(f"\n── {n_calls}-call averages ──")
for fn_name, t_sum in timing_accum.items():
    avg_ms = (t_sum / n_calls) * 1000
    print(f"  {fn_name:<25} {avg_ms:.3f} ms/call  (total {t_sum:.3f}s)")

In [ ]:
# ── Chart inputs (edit here) ──────────────────────────────────────────────
asset            = "btcusdt"
look_back        = 1440          # candles in the look-back window
look_ahead       = 240           # candles in the look-ahead window
k                = 100.0         # price scale factor (ratio 0.01 → 1.0)
bins_count       = 200           # KDE histogram bins over [-1, 1]
bandwidth        = 5             # kernel half-width in bins
kernel_type      = "Triangular"  # "Triangular" | "Epanechnikov" | "Uniform"
kde_ignore_borders = True        # exclude clipped prices (±1.0) from histogram
dt_str           = "2025-12-12 20:00:00"   # target anchor minute (UTC)
# ──────────────────────────────────────────────────────────────────────────

from strategies.lbla_n_vp.lbla_n_vp import lookback_lookahead_normalized_vp

try:
    data
except NameError:
    data = lookback_lookahead_normalized_vp(
        asset=asset, look_back=look_back, look_ahead=look_ahead,
        datetime=dt_str, k=k, bins_count=bins_count,
        bandwidth=bandwidth, kernel_type=kernel_type,
        kde_ignore_borders=kde_ignore_borders,
    )

In [ ]:
from strategies.lbla_n_vp.lbla_n_vp_chart import draw_chart_vp

draw_chart_vp(data)